# Sinh mô tả ngoài và can thiệp văn bản

Dùng GPT-4o-mini để sinh mô tả ảnh, diễn đạt lại mô tả hoặc sửa một chi tiết có chủ đích. API key được đọc khi chạy ô thực thi.

In [ ]:
import os
import sys
from pathlib import Path

source_root = Path(os.environ.get("MEDVQA_SOURCE_ROOT", ".")).resolve()
if not (source_root / "medvqa").is_dir():
    raise FileNotFoundError("Set MEDVQA_SOURCE_ROOT to the extracted source directory.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

## Cấu hình thí nghiệm

Chọn tác vụ sinh mô tả hoặc sửa văn bản, rồi khai báo thư mục đầu vào và nơi lưu kết quả. Dùng `PILOT` để kiểm tra kỹ thuật trước khi chạy `TEST`.

In [ ]:
from pathlib import Path

MODE = "PILOT"
TASK = "OBS"
RUN_ID = "pilot_obs_G"
PLATFORM = "colab"
DESIGN = "MAIN"
INPUT_ROOT = None
LOCK_FILE = None
RUN_PARAMS_FILE = None
PRIOR_OUTPUT_DIRS = []
OUT_ROOT = None
SHARD_TIME_LIMIT_S = None
SHARD_INDEX, NUM_SHARDS = (0, 1)
MAX_ITEMS_THIS_SESSION = None
OBS_MAX_TOKENS = 900
ARMS = ["ARM-GEN", "ARM-ANA", "ARM-GRD"]
SOURCE_OBS_DIRS = []
INT_TYPES = None
EDITOR = "EDIT-G"
R_GPT_LATE_ACTIVATED = False
EDIT_ALT_PRECISION = "PREC-0"
BACKEND = "openai"
STUB_FAIL_FIRST, STUB_FAIL_STATUS, STUB_EMPTY_IDS = (0, 503, ())
RETRY_WAIT_S = 60
ACK_CONTENT_ERRORS = []
INSTALL = False
MODULE_DIR = source_root

## Thực thi và lưu kết quả

In [ ]:
import sys
import time
from collections import Counter

sys.path.insert(0, str(MODULE_DIR))

In [ ]:
from medvqa import common as U
from medvqa import execution as RUN

PLATFORM = PLATFORM or "colab"
PLATFORM_CFG = RUN.platform_setup(PLATFORM)
INPUT_ROOT = Path(INPUT_ROOT or PLATFORM_CFG["input_root"])
OUT_ROOT = Path(OUT_ROOT or PLATFORM_CFG["out_root"])
SHARD_TIME_LIMIT_S = SHARD_TIME_LIMIT_S or PLATFORM_CFG["time_limit_s"]
LOCK_FILE = Path(LOCK_FILE or INPUT_ROOT / "." / "locked_inputs.sha256")

In [ ]:
from medvqa import conditions as CL

INT_TYPES = INT_TYPES or CL.edit_types_for(DESIGN)
RUN_PARAMS_FILE = Path(RUN_PARAMS_FILE or INPUT_ROOT / "." / "run_params.json")
ENV = "ENV-1" if EDITOR == "EDIT-ALT" else "ENV-API"
RUN.verify_run_params_first(LOCK_FILE, RUN_PARAMS_FILE, MODE)
platform_report = RUN.check_platform(
    "PHI" if EDITOR == "EDIT-ALT" else "G", PLATFORM, MODE, RUN_PARAMS_FILE
)
RUN.check_design(DESIGN, MODE, RUN_PARAMS_FILE)
ENV_PINS = RUN.env_pins_for(MODE, RUN_PARAMS_FILE, ENV)
if INSTALL:
    RUN.install_env(ENV, ENV_PINS)
env_report = RUN.verify_env(ENV, ENV_PINS, MODE)
print("environment:", env_report)

In [ ]:
from medvqa import interventions as I
from medvqa import models as B
from medvqa import observers as G
from medvqa import prompts as P
from medvqa import routing as R

PREP = INPUT_ROOT / "prep_out"
PROMPTS = INPUT_ROOT / "resources/protocol" / "prompts"
U.assert_no_sealed_inputs(
    [INPUT_ROOT] + list(PRIOR_OUTPUT_DIRS) + list(SOURCE_OBS_DIRS)
)
if EDITOR == "EDIT-ALT" and (not R_GPT_LATE_ACTIVATED):
    raise ValueError("EDIT-ALT requires the R-GPT-LATE configuration")
if EDITOR == "EDIT-G" and BACKEND == "openai":
    G.check_deadline()
    send = G.openai_sender(U.get_secret("OPENAI_API_KEY"))
elif BACKEND == "stub":
    send = G.stub_sender(
        fail_first=STUB_FAIL_FIRST,
        fail_status=STUB_FAIL_STATUS,
        empty_ids=STUB_EMPTY_IDS,
    )
sleep = (lambda s: None) if BACKEND == "stub" else time.sleep
STOP_ERROR = None

In [ ]:
COHORTS = RUN.cohorts_for(MODE)
lock_files = RUN.prompt_lock_files(
    INPUT_ROOT, ["templates/intervention_editor_prompts.md"]
)
lock_files.update(
    RUN.prep_lock_files(
        INPUT_ROOT,
        [
            "route_metadata.json",
            "route_pairs.json",
            "observer_prompt_manifest.json",
            "images_index.json",
        ]
        + [f"cohorts/{c}.jsonl" for c in COHORTS],
    )
)
if MODE == "TEST":
    lock_files["run_params.json"] = RUN_PARAMS_FILE
lock_report = U.verify_locked_inputs(LOCK_FILE, lock_files, MODE)
if MODE == "TEST":
    rp = U.read_json(RUN_PARAMS_FILE)
    if TASK == "OBS" and RUN.rp_get(rp, "obs_max_new_tokens") != OBS_MAX_TOKENS:
        raise U.LockError(
            "OBS_MAX_TOKENS differs from locked run_params.obs_max_new_tokens"
        )
    if (
        EDITOR == "EDIT-ALT"
        and RUN.rp_get(rp, "precision", "PHI") != EDIT_ALT_PRECISION
    ):
        raise U.LockError("EDIT_ALT_PRECISION differs from locked run_params.json")
rows = RUN.load_cohort_rows(PREP, COHORTS)
nids = sorted({r["neutral_id"] for r in rows})
cohort_of = {r["neutral_id"]: r["cohort"] for r in rows}
rm = U.read_json(PREP / "route_metadata.json")["records"]
rp_table = U.read_json(PREP / "route_pairs.json")
routing = {n: R.route_for(n, rm[n], rp_table) for n in nids}
bank = P.ObserverPromptBank(PROMPTS)
session = U.ShardSession(
    f"obs_gpt_{TASK}",
    RUN_ID,
    MODE,
    OUT_ROOT,
    SHARD_INDEX,
    NUM_SHARDS,
    PRIOR_OUTPUT_DIRS,
    time_limit_s=SHARD_TIME_LIMIT_S,
    max_items=MAX_ITEMS_THIS_SESSION,
)
ART = OUT_ROOT / RUN_ID / "artifacts"
print(MODE, TASK, "| images:", len(nids), "| locked ok:", len(lock_report["files"]))

In [ ]:
if TASK == "OBS":
    manifest = U.read_json(PREP / "observer_prompt_manifest.json")["entries"]
    allowed = bank.allowed_texts(sorted({r["route_key"] for r in routing.values()}))
    locked_lines = bank.locked_lines()
    questions = sorted({r["question"] for r in rows})
    options = sorted({o["text"] for r in rows if r["options"] for o in r["options"]})
    order = G.call_order(nids, ARMS, "GPT-v1")
    U.write_json_once(
        ART / "gpt_call_order.json",
        {"salt": "GPT-v1", "order": [f"{n}|{a}" for n, a in order]},
    )
    keys = [f"OBS|OBS-G|{a}|{n}" for n, a in order]
    shard_keys = U.plan_shard(keys, NUM_SHARDS, SHARD_INDEX)
    run_cfg_sha = RUN.write_run_config(
        session,
        {
            "notebook": "obs_gpt",
            "platform": dict(PLATFORM_CFG, lock=platform_report),
            "design": DESIGN,
            "task": TASK,
            "mode": MODE,
            "model": G.GPT_MODEL,
            "temperature": 0,
            "seed": G.GPT_SEED,
            "max_tokens": OBS_MAX_TOKENS,
            "detail": "low",
            "image_position": "before text",
            "attempts": 3,
            "retry_wait_s": RETRY_WAIT_S,
            "transport_max_retries": 5,
            "breaker_consecutive": G.BREAKER_CONSECUTIVE,
            "ack_content_errors": sorted(ACK_CONTENT_ERRORS),
            "call_order_sha256": U.sha256_file(ART / "gpt_call_order.json"),
            "env": U.environment_info(),
            "env_pins": env_report,
            "locked_inputs": lock_report,
        },
    )
    done, _ = session.done_keys()
    todo = [
        (k, p) for k, p in zip(keys, order) if k in set(shard_keys) and k not in done
    ]
    print(
        "calls planned:",
        len(keys),
        "| this shard:",
        len(shard_keys),
        "| to do:",
        len(todo),
    )
    buffer = G.ObsErrorBuffer(
        session.write,
        acknowledged=ACK_CONTENT_ERRORS,
        pending_log=session.out_dir / "breaker_pending_keys.json",
    )
    try:
        for key, (nid, arm) in todo:
            if session.should_stop():
                break
            prompt, source = bank.prompt(arm, routing[nid]["routes"])
            P.check1_observer_prompt(
                prompt, allowed, questions, options, locked_lines=locked_lines
            )
            if manifest[f"{arm}|{nid}"]["prompt_sha256"] != U.sha256_text(prompt):
                raise U.IntegrityError("prompt differs from prep manifest")
            png = (PREP / "images" / f"{nid}.png").read_bytes()
            payload = G.obs_payload(prompt, png, OBS_MAX_TOKENS)
            G.check11(payload, expect_image=True)
            U.append_jsonl(
                session.out_dir / "payload_log.jsonl",
                {"key": key, "payload": G.loggable(payload)},
            )
            t0 = time.time()
            status, resp, attempts = G.call_observation(
                send, payload, wait_s=RETRY_WAIT_S, sleep=sleep
            )
            ch = (resp or {}).get("choices", [{}])[0]
            text = (ch.get("message") or {}).get("content") or ""
            rec = session.base("OBS-v1", key)
            rec.update(
                {
                    "observer": "OBS-G",
                    "arm": arm,
                    "neutral_id": nid,
                    "cohort": cohort_of[nid],
                    "route_key": routing[nid]["route_key"],
                    "routes": routing[nid]["routes"],
                    "route_gaps": routing[nid]["gaps"],
                    "prompt_source": source,
                    "prompt_sha256": U.sha256_text(prompt),
                    "status": status,
                    "text": text if status == "ok" else "",
                    "finish_reason": ch.get("finish_reason") if resp else "error",
                    "truncated": ch.get("finish_reason") == "length",
                    "n_output_tokens": ((resp or {}).get("usage") or {}).get(
                        "completion_tokens"
                    ),
                    "attempts": attempts,
                    "api": G.api_meta(resp),
                    "quality": U.quality_flags(text),
                    "gen": {
                        "temperature": 0,
                        "seed": G.GPT_SEED,
                        "max_tokens": OBS_MAX_TOKENS,
                        "detail": "low",
                    },
                    "timing_s": round(time.time() - t0, 3),
                }
            )
            buffer.add(rec)
        buffer.flush()
    except (G.SystemAPIError, G.DeadlinePassed) as exc:
        STOP_ERROR = exc
        session.stopped_reason = (
            "deadline_passed"
            if isinstance(exc, G.DeadlinePassed)
            else "system_api_error"
        )
        print("STOPPED:", type(exc).__name__, str(exc)[:200])

In [ ]:
if TASK == "EDIT":
    ED = I.EditorPrompts(PROMPTS / "templates" / "intervention_editor_prompts.md")
    P.check5_intervention_signature(
        [
            I.corruption_plan,
            ED.para_prompt,
            ED.corr_prompt,
            I.para_checks,
            I.corr_checks,
            I.shuf_assign,
            I.empty_observation,
            I.gray_image,
        ]
    )
    src = RUN.load_records(SOURCE_OBS_DIRS, "OBS-v1")

    def src_obs(observer, nid):
        r = src.get(f"OBS|{observer}|ARM-GRD|{nid}")
        if r is None:
            raise U.IntegrityError(
                (
                    "missing source observation "
                    f"{observer}"
                    " GRD "
                    f"{nid}"
                    ": run observations first"
                )
            )
        return r

    need = {"INT-PARA": "OBS-Q", "INT-CORR": "OBS-Q", "INT2-PARA": "OBS-G"}
    for t in INT_TYPES:
        for n in nids:
            src_obs(need[t], n)
    plan, plan_sha = (None, None)
    if "INT-CORR" in INT_TYPES:
        plan = I.corruption_plan({n: src_obs("OBS-Q", n)["text"].strip() for n in nids})
        U.write_json_once(ART / "corruption_plan.json", plan)
        plan_sha = U.sha256_file(ART / "corruption_plan.json")
        print(
            "corruption_plan.json sha256:",
            plan_sha,
            "| counts:",
            plan["counts"],
            "| no feasible:",
            len(plan["no_feasible"]),
        )
    order = G.call_order(nids, INT_TYPES, "EDIT-v1")
    U.write_json_once(
        ART / "edit_call_order.json",
        {"salt": "EDIT-v1", "order": [f"{n}|{t}" for n, t in order]},
    )
    keys = [f"EDIT|{t}|{n}" for n, t in order]
    shard_keys = U.plan_shard(keys, NUM_SHARDS, SHARD_INDEX)
    alt = None
    if EDITOR == "EDIT-ALT":
        alt = (
            B.StubVLM("PHI")
            if BACKEND == "stub"
            else B.HFTextBackend("PHI", EDIT_ALT_PRECISION)
        )
    run_cfg_sha = RUN.write_run_config(
        session,
        {
            "notebook": "obs_gpt",
            "platform": dict(PLATFORM_CFG, lock=platform_report),
            "design": DESIGN,
            "task": TASK,
            "mode": MODE,
            "editor": EDITOR,
            "model": G.GPT_MODEL if EDITOR == "EDIT-G" else alt.identity,
            "temperature": 0,
            "seeds": list(G.EDIT_SEEDS),
            "max_tokens": G.EDIT_MAX_TOKENS,
            "corruption_plan_sha256": plan_sha,
            "source_obs_dirs": [str(d) for d in SOURCE_OBS_DIRS],
            "corr_synonyms": I.CORR_SYNONYMS,
            "env": U.environment_info(),
            "env_pins": env_report,
            "locked_inputs": lock_report,
        },
    )
    done, _ = session.done_keys()
    todo = [
        (k, p) for k, p in zip(keys, order) if k in set(shard_keys) and k not in done
    ]
    print("edits planned:", len(keys), "| to do:", len(todo))
    try:
        for key, (nid, t) in todo:
            if session.should_stop():
                break
            s = src_obs(need[t], nid)
            obs = s["text"].strip()
            rec = session.base("EDIT-v1", key)
            rec.update(
                {
                    "int_type": t,
                    "editor": EDITOR,
                    "neutral_id": nid,
                    "cohort": cohort_of[nid],
                    "source_obs_key": s["key"],
                    "source_obs_text_sha256": U.sha256_text(obs),
                    "attempts": [],
                }
            )
            if t == "INT-CORR":
                a = plan["assignments"].get(nid)
                if a is None:
                    rec.update(
                        {
                            "status": "int_missing",
                            "final_text": None,
                            "reason": "no_feasible_line",
                        }
                    )
                    session.write(rec)
                    continue
                rec["corr"] = {k: a[k] for k in ("edit_type", "item_no", "line_no")}
                prompt = ED.corr_prompt(obs, a["edit_type"], a["line_no"])
            else:
                prompt = ED.para_prompt(obs)
            final = None
            for seed in G.EDIT_SEEDS if EDITOR == "EDIT-G" else (None,):
                if EDITOR == "EDIT-G":
                    payload = G.edit_payload(prompt, seed)
                    G.check11(payload, expect_image=False)
                    U.append_jsonl(
                        session.out_dir / "payload_log.jsonl",
                        {"key": key, "seed": seed, "payload": G.loggable(payload)},
                    )
                    raw, meta = G.call_editor(
                        send, payload, wait_s=RETRY_WAIT_S, sleep=sleep
                    )
                else:
                    g = alt.generate_text(prompt, G.EDIT_MAX_TOKENS)
                    raw, meta = (g["text"], {"model": B.MODELS["PHI"]["id"]})
                if raw is None:
                    rec["attempts"].append(
                        {
                            "seed": seed,
                            "raw_text": None,
                            "api": meta,
                            "checks": {},
                            "passed": False,
                        }
                    )
                    continue
                if t == "INT-CORR":
                    checks, edited = I.corr_checks(
                        obs, raw, rec["corr"]["line_no"], rec["corr"]["edit_type"]
                    )
                else:
                    checks, edited = (I.para_checks(obs, raw.strip()), raw.strip())
                passed = bool(checks) and all(checks.values())
                rec["attempts"].append(
                    {
                        "seed": seed,
                        "raw_text": raw,
                        "api": meta,
                        "checks": checks,
                        "passed": passed,
                    }
                )
                if passed:
                    final = edited
                    break
            rec.update(
                {
                    "status": "ok" if final is not None else "int_missing",
                    "final_text": final,
                }
            )
            session.write(rec)
    except (G.SystemAPIError, G.DeadlinePassed) as exc:
        STOP_ERROR = exc
        session.stopped_reason = (
            "deadline_passed"
            if isinstance(exc, G.DeadlinePassed)
            else "system_api_error"
        )
        print("STOPPED:", type(exc).__name__, str(exc)[:200])

In [ ]:
rep = session.integrity(
    shard_keys,
    {
        "checks": {
            "check1": "pass" if TASK == "OBS" else "n/a",
            "check5": "pass" if TASK == "EDIT" else "n/a",
            "check7": lock_report["verified"],
            "check11": "pass",
        },
        "run_config_sha256": run_cfg_sha,
        "stop_error": repr(STOP_ERROR) if STOP_ERROR else None,
        "locked_inputs": {
            "verified": lock_report["verified"],
            "unlisted_used": lock_report["unlisted_used"],
        },
    },
)
recs = U.read_jsonl(session.records)
print(
    {
        k: rep[k]
        for k in (
            "stopped_reason",
            "done_this_session",
            "remaining_in_shard",
            "duplicate_keys",
        )
    },
    "| status:",
    dict(Counter((r["status"] for r in recs))),
)
if isinstance(STOP_ERROR, G.DeadlinePassed):
    raise G.DeadlinePassed(
        (
            "GPT deadline passed: do NOT call GPT again. "
            "R-GPT-LATE: OBS-Q replaces"
            " OBS-G as external observer; EDITOR='EDIT-ALT' "
            "with R_GPT_LATE_ACTIVATED=True for ALL LLM "
            "interventions."
        )
    )
if STOP_ERROR is not None:
    raise G.SystemAPIError(
        (
            "Session stopped by the API circuit breaker; fix "
            "the cause (key/quota/network) and resume the "
            "same RUN_ID."
        )
    )